# Validate Knowledge Graph Repository

Simple check on kg repo capabilities:
- start with seed id for a given paragraph (mentions greek polis)
- find entities in that paragraph
- find 1st order hops from those entities
- create graph with pyvis

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

In [2]:
from history_book.database.config.database_config import WeaviateConfig
from history_book.database.repositories.book_repository import BookRepositoryManager
from history_book.utils import print_with_wrapping

import pandas as pd

In [3]:

config = WeaviateConfig.from_environment()
mgr = BookRepositoryManager(config)

# Quick counts
print(f"KG Entities:       {mgr.kg_entities.count()}")
print(f"KG Relationships:  {mgr.kg_relationships.count()}")
print(f"KG Graphs:         {mgr.kg_graphs.count()}")

# What graphs do we have?
for g in mgr.kg_graphs.list_all():
    print(f"\n  {g.name} ({g.graph_type}): {g.entity_count} entities, {g.relationship_count} rels, chapters={g.book_chapters}")

KG Entities:       1026
KG Relationships:  1996
KG Graphs:         3

  book3_ch4_5 (book): 487 entities, 998 rels, chapters=['3:4', '3:5']

  book3_ch4 (chapter): 195 entities, 351 rels, chapters=['3:4']

  book3_ch5 (chapter): 344 entities, 647 rels, chapters=['3:5']


In [3]:
# Which graph has this paragraph? Use the merged book graph.
graph_name = "book3_ch0-7"


In [4]:
mgr.kg_entities.search_entities(query_text="Parthia", graph_name=graph_name)

[]

In [5]:
search_res = mgr.kg_entities.search_entities_hybrid(query_text="Parthia", graph_name=graph_name, limit=10)
search_res

[(KGEntity(id='b8b77f5f-8124-4fbc-bcdf-e07d37c24b16', graph_name='book3_ch0-7', name='Ctesiphon', entity_type='place', aliases=[], descriptions=['Capital of Parthia conquered by Trajan.', 'City where Ardashir was crowned king after killing the last king of Parthia.'], occurrence_count=2, book_indices=[3], source_book_chapters=['3:4', '3:5'], source_pages=[346, 389], source_paragraph_ids=['4fe58a41-098d-4d03-bbe0-6867e560a190', '1cfb6550-4c6e-4c63-a181-7dc7a3a15e46'], merged_from_count=2),
  0.5),
 (KGEntity(id='c9dcabf6-21d8-4f49-9061-759f1b7971a7', graph_name='book3_ch0-7', name='Roman legions', entity_type='concept', aliases=['legions'], descriptions=["The military units of Rome that became increasingly professional and were given 'eagles' as standards by Marius.", 'The core military units of the Roman army, numbering twenty-eight after AD 9.'], occurrence_count=2, book_indices=[3], source_book_chapters=['3:4'], source_pages=[334, 359], source_paragraph_ids=['c5114908-0cf5-4856-9b39-

In [6]:
parthia_entity = search_res[6][0]
print(f"Entity: {parthia_entity.name} ({parthia_entity.entity_type})")

Entity: Parthian Empire (polity)


In [7]:
parthia_entity.__dict__

{'id': 'dfc14c46-692a-4c69-97b9-475fd1d0b518',
 'graph_name': 'book3_ch0-7',
 'name': 'Parthian Empire',
 'entity_type': 'polity',
 'aliases': ['Parthians', 'Parthia', 'Parthia (Parthian Empire)'],
 'descriptions': ['Empire established by Mithridates I, stretching from Bactria in the east to Babylonia in the west.',
  'An Iranian nomadic people who created political unity in Iran and Mesopotamia and were known for mounted archery.',
  'A kingdom in the east that emerged as a new threat and an important satrapy south-east of the Caspian Sea.',
  'A polity located on the Euphrates, facing a new power from the west.',
  'A major power with dynastic instability and contested control over Armenia.',
  'A great power in Asia with a strong army including mounted archers and cataphracts.',
  'Eastern polity against which Caesar planned a great campaign.',
  'A polity that harassed Syria and encouraged unrest among Palestinian Jews, involved in civil war politics against Rome.',
  'One of the g

In [8]:
seed_id = parthia_entity.source_paragraph_ids[0]
print(f"\nSource paragraph ID: {seed_id}")


Source paragraph ID: bedf83eb-0968-46f4-9f55-64670a1b43f3


In [9]:

# Step 1: Find entities in the seed paragraph
seed_entities = mgr.kg_entities.find_by_paragraph(seed_id, graph_name)
print(f"Entities in paragraph {seed_id[:8]}...:")
for e in seed_entities:
    print(f"  {e.name} ({e.entity_type}) — aliases={e.aliases}, occurrences={e.occurrence_count}")

Entities in paragraph bedf83eb...:
  Parthian Empire (polity) — aliases=['Parthians', 'Parthia', 'Parthia (Parthian Empire)'], occurrences=9
  Roman Senate (polity) — aliases=['Senate'], occurrences=19
  Julius Caesar (person) — aliases=['Julius', 'Caesar'], occurrences=10


In [10]:
# Step 2: Find 1st-order hop relationships from seed entities
seed_entity_ids = [e.id for e in seed_entities]
rels = mgr.kg_relationships.find_by_entities(seed_entity_ids, graph_name)
print(f"Relationships involving seed entities: {len(rels)}")
for r in rels:
    print(f"  {r.source_entity_name} --[{r.relation_type}]--> {r.target_entity_name}  (temporal: {r.temporal_context or '-'})")

Relationships involving seed entities: 76
  Parthia --[conquered]--> Seleucid kingdom  (temporal: third century BC)
  Parthia --[influenced]--> Silk Road  (temporal: -)
  Parthians --[part_of]--> Iran  (temporal: -)
  Parthians --[part_of]--> Mesopotamia  (temporal: -)
  Parthians --[influenced]--> Seleucids  (temporal: -)
  Mithridates I --[ruled]--> Parthian empire  (temporal: second century BC)
  Mithridates II --[ruled]--> Parthian empire  (temporal: second century BC)
  Parthia --[opposed]--> Rome  (temporal: -)
  Roman Senate --[part_of]--> Rome  (temporal: -)
  Roman Senate --[opposed]--> Roman People  (temporal: -)
  Senate --[part_of]--> Roman Republic  (temporal: 300 BC)
  Senate --[ruled]--> Roman Republic  (temporal: 300 BC)
  Senate --[part_of]--> patricians  (temporal: 300 BC)
  Senate --[part_of]--> plebs  (temporal: 300 BC)
  consul --[part_of]--> Senate  (temporal: -)
  consuls --[ruled]--> Senate  (temporal: end of the sixth century BC)
  quaestors --[part_of]--> Sena

In [11]:
# Step 3: Collect all connected entities (1-hop neighbors)
neighbor_ids = set()
for r in rels:
    neighbor_ids.add(r.source_entity_id)
    neighbor_ids.add(r.target_entity_id)

# Fetch neighbor entities not already in seed set
new_ids = neighbor_ids - set(seed_entity_ids)
all_entities = {e.id: e for e in seed_entities}
for nid in new_ids:
    entity = mgr.kg_entities.get_by_id(nid)
    if entity:
        all_entities[nid] = entity

print(f"Seed entities: {len(seed_entities)}")
print(f"1-hop neighbors: {len(new_ids)}")
print(f"Total subgraph entities: {len(all_entities)}")

Seed entities: 3
1-hop neighbors: 44
Total subgraph entities: 47


In [12]:
# Step 4: Visualize subgraph with PyVis
from pyvis.network import Network

TYPE_COLORS = {
    "person": "#4e79a7",
    "polity": "#f28e2b",
    "place": "#e15759",
    "event": "#76b7b2",
    "concept": "#59a14f",
}

net = Network(height="600px", width="100%", notebook=True, cdn_resources="in_line")
net.barnes_hut(gravity=-3000, spring_length=150)

# Add nodes — seed entities are larger
for eid, e in all_entities.items():
    is_seed = eid in seed_entity_ids
    color = TYPE_COLORS.get(e.entity_type, "#999999")
    net.add_node(
        eid,
        label=e.name,
        title=f"{e.name}\nType: {e.entity_type}\nAliases: {', '.join(e.aliases) if e.aliases else '-'}\nOccurrences: {e.occurrence_count}",
        color=color,
        size=25 if is_seed else 15,
        borderWidth=3 if is_seed else 1,
    )

# Add edges
for r in rels:
    if r.source_entity_id in all_entities and r.target_entity_id in all_entities:
        net.add_edge(
            r.source_entity_id,
            r.target_entity_id,
            title=f"{r.relation_type}\n{r.temporal_context or ''}",
            label=r.relation_type,
            font={"size": 9},
        )

net.show("kg_subgraph.html")
print(f"Graph saved to kg_subgraph.html — {len(all_entities)} nodes, {len(rels)} edges")

kg_subgraph.html
Graph saved to kg_subgraph.html — 47 nodes, 76 edges


# Inspect largest entities

In [18]:
top_entities = mgr.kg_entities.find_by_criteria(criteria={"graph_name": graph_name}, sort_by="occurrence_count", ascending=False, limit=10)

In [20]:
for entity in top_entities:
    print(f"{entity.name} ({entity.entity_type}) — occurrences: {entity.occurrence_count}, aliases: {entity.aliases}")

Roman Empire (polity) — occurrences: 121, aliases: ['republic', 'Roman administration', 'Rome (Roman Republic)', 'Rome (Roman Republic / Roman Empire)', 'early republic', 'Rome', 'Roman society', 'eastern empire', 'Roman government', 'Romans', 'Rome (Roman Republic / early Roman state)', 'old empire', 'The republic', 'Romans (Roman Empire)', 'Empire in the west', 'Roman authorities', 'western empire', 'empire', 'Empire', 'Eastern Empire']
Greeks (polity) — occurrences: 45, aliases: ['Greek society', 'Greek thought', 'ancient Greece', 'Greek', 'Hellenistic kingdoms', 'Greek literature', 'Greek world', 'archaic Greece', 'city-states of Greece', 'classical Greece', 'Hellenes', 'these states', 'Greek cities']
Persia (polity) — occurrences: 40, aliases: ['Persians', 'Achaemenid Persians (Achaemenid Empire)', 'Persian power', 'Achaemenids']
Rome (place) — occurrences: 36, aliases: ['church of Rome']
Athens (polity) — occurrences: 29, aliases: ['Athenians']
Christianity (concept) — occurrence

## Investigate: Persia relationships

In [25]:
ent_persia = top_entities[2]

In [27]:
rels_persia = mgr.kg_relationships.find_by_entities(entity_ids=[ent_persia.id], graph_name=graph_name)

In [28]:
for rel in rels_persia:
    print(f"{rel.source_entity_name} --[{rel.relation_type}]--> {rel.target_entity_name}  (temporal: {rel.temporal_context or '-'})")   

Persians --[conquered]--> Egypt  (temporal: 525)
Persia --[opposed]--> Lydia  (temporal: -)
Persia --[conquered]--> Peloponnese  (temporal: -)
Darius --[ruled]--> Persia  (temporal: -)
Greece --[opposed]--> Persia  (temporal: -)
Persia --[conquered]--> Lydia  (temporal: about 540 BC)
Persia --[conquered]--> Ionia  (temporal: -)
Cyrus --[ruled]--> Persia  (temporal: -)
Greek cities in Asia --[opposed]--> Persia  (temporal: early fifth century BC)
Persians --[conquered]--> Eretria  (temporal: 490 BC)
Persians --[opposed]--> Greeks  (temporal: -)
Athenians --[opposed]--> Persians  (temporal: 490 BC)
Persians --[conquered]--> Mount Athos  (temporal: -)
Persians --[participated_in]--> Marathon  (temporal: 490 BC)
Sparta --[opposed]--> Persia  (temporal: -)
Athens --[opposed]--> Persia  (temporal: -)
Persian army --[part_of]--> Persians  (temporal: 480 BC)
Greek civilization --[opposed]--> Persians  (temporal: -)
Delian League --[opposed]--> Persia  (temporal: -)
Spartans --[allied_with]--> 

# Investigate: two frances

In [30]:
graph_name = "book7_ch0-6"

## text search

In [33]:
mgr.kg_entities.search_entities_hybrid(query_text="France", graph_name=graph_name, limit=20)

[(KGEntity(id='0e922b0e-e4e6-43eb-a5e1-4786fa82cc7f', graph_name='book7_ch0-6', name='France', entity_type='polity', aliases=['Russian state', 'Russians', 'French', 'Russia', 'tsarist state', 'Great Britain', 'Allied governments'], descriptions=["A major polity comprising a quarter of Europe's population in 1900.", 'A country in Europe with slow economic and social progress, experiencing revolutionary movements in the early 20th century.', 'A European great power involved in a war with Turkey in 1876.', 'A great dynastic empire facing internal problems and political revolution in 1905.', 'A major polity undergoing economic advance and social challenges in the early twentieth century.', 'A colonial power that came into conflict with the United Kingdom in various global locations.', 'A nation defeated militarily by Japan.', 'A major European state with an older political constitution than the United States in 1914.', 'A great state involved in the European balance of power before 1914.',

## inspect in top entities

In [34]:
top_entities = mgr.kg_entities.find_by_criteria(criteria={"graph_name": graph_name}, sort_by="occurrence_count", ascending=False, limit=10)

In [35]:
top_entities

[KGEntity(id='5ddc709f-2dc1-43ea-828b-55556b30ca05', graph_name='book7_ch0-6', name='Germany', entity_type='polity', aliases=['German republic', 'German army', 'German Reich', 'Bismarck’s creation', 'German forces', 'Central Powers', 'democratic German republic', 'new Germany', 'Germans', 'German', 'Austria-Hungary', 'united Germany'], descriptions=['One of the most stricken countries in Europe with about 7½ million dwellings destroyed.', 'An industrial European country whose economic capacity was suppressed by the Allies and was divided after 1918.', 'A state whose colonial interests were promoted by Bismarck in response to public opinion.', 'A great power that acquired Alsace and Lorraine in 1871 and whose nationalism under Wilhelm II caused alarm in Europe.', 'A nation where anti-clericalism and priest-baiting were significant political issues in the late 19th century.', 'Republic in Germany dominated by socialists after defeat.', 'A new state formed less than forty years before 190

### First occurance

In [45]:
ent_france_1 = top_entities[5]

In [46]:
ent_france_1.__dict__

{'id': '0e922b0e-e4e6-43eb-a5e1-4786fa82cc7f',
 'graph_name': 'book7_ch0-6',
 'name': 'France',
 'entity_type': 'polity',
 'aliases': ['Russian state',
  'Russians',
  'French',
  'Russia',
  'tsarist state',
  'Great Britain',
  'Allied governments'],
 'descriptions': ["A major polity comprising a quarter of Europe's population in 1900.",
  'A country in Europe with slow economic and social progress, experiencing revolutionary movements in the early 20th century.',
  'A European great power involved in a war with Turkey in 1876.',
  'A great dynastic empire facing internal problems and political revolution in 1905.',
  'A major polity undergoing economic advance and social challenges in the early twentieth century.',
  'A colonial power that came into conflict with the United Kingdom in various global locations.',
  'A nation defeated militarily by Japan.',
  'A major European state with an older political constitution than the United States in 1914.',
  'A great state involved in the

In [47]:
rels_france_1 = mgr.kg_relationships.find_by_entities(entity_ids=[ent_france_1.id], graph_name=graph_name)

In [48]:
for rel in rels_france_1:
    print(f"{rel.source_entity_name} --[{rel.relation_type}]--> {rel.target_entity_name}  (temporal: {rel.temporal_context or '-'})   (desc: {rel.description or '-'})")   

Russia --[part_of]--> Europe  (temporal: 1900)   (desc: Russians made up a quarter of Europe's population in 1900.)
revolution in 1905 --[part_of]--> Russia  (temporal: 1905)   (desc: The 1905 revolution occurred within Russia.)
Japanese --[conquered]--> Russia  (temporal: -)   (desc: Japan defeated Russia in war, shaking the regime's confidence and contributing to the 1905 revolution.)
Russia --[opposed]--> Turkey  (temporal: 1876)   (desc: Russia and Turkey came to blows in 1876, marking the last war between European great powers in the nineteenth century.)
Russia --[participated_in]--> 1905 Revolution  (temporal: 1905)   (desc: Russia experienced a political revolution in 1905, marking deep internal change.)
Alexander II --[influenced]--> Russia  (temporal: late 19th century)   (desc: Alexander II's reforms promised liberal change in Russia but were undermined by autocracy and terrorism.)
Russia --[conquered]--> serfdom  (temporal: -)   (desc: Serfdom was abolished in Russia, removi

### Second occurance

In [49]:
ent_france_2 = top_entities[6]

In [50]:
ent_france_2.__dict__

{'id': '4286e4e7-a53e-4856-a3ff-016337a38c29',
 'graph_name': 'book7_ch0-6',
 'name': 'France',
 'entity_type': 'polity',
 'aliases': ['French', 'French Empire', 'French empire'],
 'descriptions': ['One of the powers with which China had unequal treaties that were swept away during the Second World War.',
  'French armed forces later admitted to Berlin administration after Soviet agreement.',
  'Colonial power contrasted with British in terms of concessions to nationalist sentiment.',
  'Colonial power in South-East Asia with colonies affected by World War II.',
  'Colonial power in Indochina attempting to maintain control after WWII.',
  'Colonial power trying to retain a diminished Vietnam and other Indochinese states within the French Union.',
  'A great power that held League of Nations mandates in Syria and Lebanon until 1939.',
  'Tried to re-establish authority in Syria and Lebanon after the war and faced nationalist movements in Algeria.',
  'French colonial possessions where D

In [51]:
rels_france_2 = mgr.kg_relationships.find_by_entities(entity_ids=[ent_france_2.id], graph_name=graph_name)

In [52]:
for rel in rels_france_2:
    print(f"{rel.source_entity_name} --[{rel.relation_type}]--> {rel.target_entity_name}  (temporal: {rel.temporal_context or '-'})   (desc: {rel.description or '-'})")   

France --[opposed]--> Germany  (temporal: post-1871)   (desc: France opposed Germany due to the loss of Alsace and Lorraine and cultivated the theme of revanche.)
France --[opposed]--> Church  (temporal: 1870-1890)   (desc: Anti-clericalism and priest-baiting were important political issues opposing the Church.)
Germany --[opposed]--> France  (temporal: -)   (desc: Germany was fighting France on one of its two fronts and had suffered heavy casualties.)
Allies --[allied_with]--> France  (temporal: 1917)   (desc: France was part of the Allies during World War I.)
Passchendaele --[participated_in]--> France  (temporal: 1917)   (desc: The battles of Passchendaele took place in France in 1917.)
French army --[part_of]--> France  (temporal: 1917)   (desc: The French army was the military force of France during World War I.)
France --[opposed]--> Germany  (temporal: -)   (desc: France was aware of the danger of German aggression and was a central actor in European security concerns.)
United S

# Check vectors

In [2]:
from history_book.database.config.database_config import WeaviateConfig
from history_book.database.repositories.book_repository import BookRepositoryManager

config = WeaviateConfig.from_environment()
mgr = BookRepositoryManager(config)

# Quick counts
print(f"KG Entities:       {mgr.kg_entities.count()}")
print(f"KG Relationships:  {mgr.kg_relationships.count()}")
print(f"KG Graphs:         {mgr.kg_graphs.count()}")

# What graphs do we have?
for g in mgr.kg_graphs.list_all():
    print(f"\n  {g.name} ({g.graph_type}): {g.entity_count} entities, {g.relationship_count} rels, chapters={g.book_chapters}")

KG Entities:       247
KG Relationships:  457
KG Graphs:         1

  book3_ch2-3 (book): 247 entities, 457 rels, chapters=['3:2', '3:3']


In [3]:
# Which graph has this paragraph? Use the merged book graph.
graph_name = "book3_ch2-3"


In [4]:
mgr.kg_entities.search_entities(query_text="Parthia", graph_name=graph_name)

[(KGEntity(id='da8c0473-13a4-4e48-959c-c69627055aaf', graph_name='book3_ch2-3', name='Parthian Empire', entity_type='polity', aliases=['Parthia (Parthian Empire)', 'Parthia'], descriptions=['Empire established by Mithridates I, stretching from Bactria in the east to Babylonia in the west.', 'An Iranian nomadic people who created political unity in Iran and Mesopotamia and were known for mounted archery.', 'A kingdom in the east that emerged as a new threat and an important satrapy south-east of the Caspian Sea.', 'A polity located on the Euphrates, facing a new power from the west.'], occurrence_count=4, book_indices=[3], source_book_chapters=['3:3'], source_pages=[312, 313], source_paragraph_ids=['01f84f6a-0902-49f7-ab19-fa9343e44aed', 'e531a0c6-ba46-4aad-8994-013584ffe079', '00e27974-33f2-4e98-aab0-e97511ccdb82', 'b9b8b555-088a-4da9-a46f-db3112ff31bc'], merged_from_count=4),
  0.7157468795776367),
 (KGEntity(id='ecf03ced-fc1e-42ef-8570-086133bfe0cf', graph_name='book3_ch2-3', name='M

In [7]:
test_rel = mgr.kg_relationships.find_by_criteria(criteria={"graph_name": graph_name, "relation_type": "influenced"}, limit=1)[0]

In [8]:
test_rel.__dict__

{'id': 'dfdc7770-9c12-47a3-83fe-d9abe2314c12',
 'graph_name': 'book3_ch2-3',
 'source_entity_id': '36f27116-2092-4d75-9074-b915a6f1561a',
 'target_entity_id': 'a8c0a709-5739-432a-a28a-54e434453d60',
 'entity_ids': ['36f27116-2092-4d75-9074-b915a6f1561a',
  'a8c0a709-5739-432a-a28a-54e434453d60'],
 'source_entity_name': 'Greece',
 'target_entity_name': 'Italy',
 'relation_type': 'influenced',
 'description': 'Greek products have been found in Italy, indicating influence through commerce.',
 'temporal_context': 'sixth century',
 'start_year': -600.0,
 'end_year': -501.0,
 'temporal_precision': 'century',
 'paragraph_id': '196d4aa2-160c-45d2-bf05-6c08f4dafc54',
 'book_index': 3,
 'chapter_index': 2,
 'page': 265}

# Check description source paragraphs

In [3]:
from history_book.database.config.database_config import WeaviateConfig
from history_book.database.repositories.book_repository import BookRepositoryManager

config = WeaviateConfig.from_environment()
mgr = BookRepositoryManager(config)

# Quick counts
print(f"KG Entities:       {mgr.kg_entities.count()}")
print(f"KG Relationships:  {mgr.kg_relationships.count()}")
print(f"KG Graphs:         {mgr.kg_graphs.count()}")

# What graphs do we have?
for g in mgr.kg_graphs.list_all():
    print(f"\n  {g.name} ({g.graph_type}): {g.entity_count} entities, {g.relationship_count} rels, chapters={g.book_chapters}")

KG Entities:       267
KG Relationships:  466
KG Graphs:         1

  book3_ch2-3 (book): 267 entities, 466 rels, chapters=['3:2', '3:3']


In [4]:
# Which graph has this paragraph? Use the merged book graph.
graph_name = "book3_ch2-3"

In [5]:
top_entities = mgr.kg_entities.find_by_criteria(criteria={"graph_name": graph_name}, sort_by="occurrence_count", ascending=False, limit=10)

In [6]:
for entity in top_entities:
    print(f"{entity.name} ({entity.entity_type}) — occurrences: {entity.occurrence_count}, aliases: {entity.aliases}")

Greece (polity) — occurrences: 32, aliases: ['Greek literature', 'Greek world', 'Greek thought', 'classical Greece', 'archaic Greece', 'Hellenes', 'Greek states', 'Greeks', 'ancient Greece', 'Greek']
Athens (polity) — occurrences: 29, aliases: ['Athenians']
Achaemenid Empire (polity) — occurrences: 22, aliases: ['Persia', 'Persian rule', 'Persian power', 'Achaemenids', 'that empire']
Alexander the Great (person) — occurrences: 15, aliases: ['Alexander', 'the Great']
Sparta (polity) — occurrences: 15, aliases: ['Spartan', 'Spartan League', 'Spartans']
Hellenistic culture (concept) — occurrences: 13, aliases: ['Hellenistic world', 'Hellenistic (Greek) civilization', 'Greek culture', 'Greek states', 'Hellenism', 'classical civilization']
Greece (place) — occurrences: 12, aliases: ['ancient Greece', 'Greek peninsula', 'Greek world']
Athens (place) — occurrences: 10, aliases: ['fifth-century Athens', 'Athens in the fifth century']
Macedon (polity) — occurrences: 9, aliases: ['Macedonian Emp

In [7]:
ent_hellen = top_entities[5]
ent_hellen.__dict__

{'id': '33f19269-7cfe-4ef8-8207-683d64cb704b',
 'graph_name': 'book3_ch2-3',
 'name': 'Hellenistic culture',
 'entity_type': 'concept',
 'aliases': ['Hellenistic world',
  'Hellenistic (Greek) civilization',
  'Greek culture',
  'Greek states',
  'Hellenism',
  'classical civilization'],
 'descriptions': ['An idea created by later ages to represent the cultural and intellectual achievements of the Greeks, especially in the fifth century BC.',
  'The name given by the Romans centuries later to the Greek-speaking peoples.',
  'The civilization that emerged in the Aegean region influenced by earlier Minoan and Mycenaean cultures.',
  'The cultural and historical achievements of the Greeks during their greatest age.',
  'A civilization consisting of many Greek states scattered over the Aegean region.',
  'A civilization noted for its early vigorous and perceptive ideas about nature, exemplified by the Ionians.',
  'A cultural period characterized by eclecticism and cosmopolitanism, blendin

In [8]:
ent_hellen.descriptions[7]

'Civilization preserving Greek tradition in science during Hellenistic times.'

In [9]:
ent_hellen.description_paragraph_ids[7]

'5ef9ce6b-d820-46c7-90bc-1b7d9d7d945d'

In [11]:
para = mgr.paragraphs.get_by_id(ent_hellen.description_paragraph_ids[7])

In [15]:
print_with_wrapping(para.text)

Hellenistic civilization preserved the Greek tradition most successfully in
science, and here Alexandria, the greatest of all Hellenistic cities, was pre-
eminent. Euclid was the greatest systematizer of geometry, defining it until the
nineteenth century, and Archimedes, famous for his practical achievements in the
construction of war-machines in Sicily, was probably Euclid’s pupil. Another
Alexandrian, Eratosthenes, was the first man to measure the size of the earth,
and a Hellenistic Greek, Aristarchus of Samos, got so far as to say that the
earth moved around the sun, though his views were set aside by contemporaries
and posterity because they could not be squared with Aristotelian physics which
stated the contrary. In hydrostatics, Archimedes made great strides (and
invented the windlass, too), but the central achievement of the Greek tradition
was always mathematical, not practical, and in Hellenistic times it reached its
apogee with the theory of conic sections and ellipses and t

In [16]:
mgr.kg_entities.find_by_paragraph(para.id, graph_name)

[KGEntity(id='a259b46e-d9e8-4722-8a62-12b3edeae8f4', graph_name='book3_ch2-3', name='Alexandria', entity_type='place', aliases=[], descriptions=['City in Egypt where Ptolemy conducted his astronomical work.', 'City with one of the two greatest libraries of the ancient world.', 'The greatest of all Hellenistic cities, pre-eminent in science.'], description_paragraph_ids=['a4a41817-b296-4985-b034-64839febfa1c', 'f39c7c88-f6e3-4e6a-9aa9-64a902236163', '5ef9ce6b-d820-46c7-90bc-1b7d9d7d945d'], occurrence_count=3, book_indices=[3], source_book_chapters=['3:2', '3:3'], source_pages=[285, 308, 309], source_paragraph_ids=['f39c7c88-f6e3-4e6a-9aa9-64a902236163', 'a4a41817-b296-4985-b034-64839febfa1c', '5ef9ce6b-d820-46c7-90bc-1b7d9d7d945d'], merged_from_count=3),
 KGEntity(id='33f19269-7cfe-4ef8-8207-683d64cb704b', graph_name='book3_ch2-3', name='Hellenistic culture', entity_type='concept', aliases=['Hellenistic world', 'Hellenistic (Greek) civilization', 'Greek culture', 'Greek states', 'Hellen

# Merge auditing

In [4]:
mgr.kg_merge_decisions.count()

1873

In [5]:
graph_name = "book3_ch4_5"

In [6]:
all_merges = mgr.kg_merge_decisions.list_all()

In [7]:
all_merges

[KGMergeDecision(id='00078226-ff33-46ce-828f-3cfcc475ac65', graph_name='book3_ch4', merge_type='root', entity1_name='New Carthage', entity1_type='place', entity1_aliases=['Cartagena'], entity1_source_graph='', entity2_name='New Carthage', entity2_type='place', entity2_aliases=[], canonical_name='New Carthage', similarity=None, reasoning='', occurrence_count_after=1, created_at=datetime.datetime(2026, 3, 15, 20, 29, 47, 665001, tzinfo=datetime.timezone.utc)),
 KGMergeDecision(id='0011c40d-c641-4ffc-b2e8-824f1bbb2a53', graph_name='book3_ch4_5', merge_type='root', entity1_name='barbarian warlords', entity1_type='person', entity1_aliases=[], entity1_source_graph='book3_ch5', entity2_name='barbarian warlords', entity2_type='person', entity2_aliases=[], canonical_name='barbarian warlords', similarity=None, reasoning='', occurrence_count_after=1, created_at=datetime.datetime(2026, 3, 15, 20, 40, 51, 789373, tzinfo=datetime.timezone.utc)),
 KGMergeDecision(id='0028838c-3398-42e6-a8d2-5d6a3f91a

In [8]:
all_merges_df = pd.DataFrame([m.__dict__ for m in all_merges])

In [9]:
all_merges_df['merge_type'].value_counts()

merge_type
root    966
rule    568
llm     339
Name: count, dtype: int64

In [24]:
all_merges_df[all_merges_df["canonical_name"]=="Julius Caesar"]

,id,graph_name,merge_type,entity1_name,entity1_type,entity1_aliases,entity1_source_graph,entity2_name,entity2_type,entity2_aliases,canonical_name,similarity,reasoning,occurrence_count_after,created_at
25,03e9a15b-985f-424f-bca1-c62a40643e8e,book3_ch4,rule,Julius Caesar,person,[],,Julius Caesar,person,[Caesar],Julius Caesar,NaN,,7,2026-03-15 20:29:47.668584+00:00
37,0606e63f-9347-4008-b95f-c156108ca699,book3_ch4,rule,Julius Caesar,person,[Caesar],,Julius Caesar,person,[Caesar],Julius Caesar,NaN,,4,2026-03-15 20:29:47.668313+00:00
230,1f8d8504-e45d-4298-9342-ecdfce35a52a,book3_ch4,rule,Caesar,person,[],,Julius Caesar,person,"[Octavian, princeps, Caesar Augustus, Caesar]",Julius Caesar,NaN,,11,2026-03-15 20:29:47.674077+00:00
512,44ec805a-418e-4c8a-89c4-0d5c0e44535c,book3_ch4,rule,Julius Caesar,person,[Julius],,Julius Caesar,person,[Caesar],Julius Caesar,NaN,,8,2026-03-15 20:29:47.668695+00:00
629,549265b6-46a7-4e54-99ff-6bafd2d0164b,book3_ch4,rule,Caesar,person,[],,Julius Caesar,person,[Caesar],Julius Caesar,NaN,,5,2026-03-15 20:29:47.668428+00:00
920,7bd1759a-e37c-4c0b-aa63-908a23da2dd8,book3_ch4,rule,Julius Caesar,person,[Caesar],,Julius Caesar,person,"[Julius, Caesar]",Julius Caesar,NaN,,9,2026-03-15 20:29:47.670053+00:00
1131,981e69fb-2b30-494e-8dec-8d1017baaaca,book3_ch4,rule,Caesar,person,[],,Augustus,person,"[Octavian, princeps, Caesar Augustus, Caesar]",Julius Caesar,NaN,,10,2026-03-15 20:29:47.671465+00:00
1268,ab3743ba-8ecd-4863-91bf-8e72d35be6b1,book3_ch4,rule,Caesar,person,[],,Julius Caesar,person,[Caesar],Julius Caesar,NaN,,2,2026-03-15 20:29:47.668127+00:00
1375,b8ec5614-9607-4c67-b36f-987bfb8ae325,book3_ch4,rule,Julius Caesar,person,[Caesar],,Julius Caesar,person,[Caesar],Julius Caesar,NaN,,6,2026-03-15 20:29:47.668492+00:00
1671,e2e25e48-ae02-4996-b551-73dfac6b47a1,book3_ch4,rule,Caesar,person,[],,Julius Caesar,person,[Caesar],Julius Caesar,NaN,,3,2026-03-15 20:29:47.668250+00:00


In [ ]:
mgr.kg_entities.count()

1026

In [29]:
mgr.kg_entities.find_by_criteria(criteria={"name": "Julius Caesar", "graph_name": "book3_ch4"}, limit=5)

[KGEntity(id='1f0031f0-6981-49a8-b7dc-16fce5b30707', graph_name='book3_ch4', name='Julius Caesar', entity_type='person', aliases=['Julius', 'Caesar'], descriptions=['Young aristocrat elected consul in 59 BC, commander of the army of Gaul, led campaigns resulting in its conquest.', 'A formidable man who wished to remain in Gaul in command of his army and province, later led his army across the River Rubicon beginning a march to Rome.', "Roman leader who marched to Spain to defeat Pompey's legions, pursued Pompey to Egypt, and defeated opposing Roman armies in Africa and Spain.", 'Roman leader who organized political support, packed the Senate with his men, was voted dictator for life, and introduced the Julian calendar.', 'Roman leader assassinated in the Senate on 15 March 44 BC at the height of his success.', 'Roman leader whose acts were confirmed after his death and was proclaimed a god.', 'Great Roman leader, great-uncle and adoptive predecessor of Octavian.', 'Great dictator whose

In [27]:
check_merges = mgr.kg_merge_decisions.find_by_entity("Julius Caesar", entity_type="person", graph_name="book3_ch4")

In [28]:
len(check_merges)

12

In [30]:
check_merges[0].__dict__

{'id': 'f5da0770-e98a-419a-8cfd-eccb0584c7ca',
 'graph_name': 'book3_ch4',
 'merge_type': 'root',
 'entity1_name': 'Julius Caesar',
 'entity1_type': 'person',
 'entity1_aliases': ['Caesar'],
 'entity1_source_graph': '',
 'entity2_name': 'Julius Caesar',
 'entity2_type': 'person',
 'entity2_aliases': [],
 'canonical_name': 'Julius Caesar',
 'similarity': None,
 'reasoning': '',
 'occurrence_count_after': 1,
 'created_at': datetime.datetime(2026, 3, 15, 20, 29, 47, 668085, tzinfo=datetime.timezone.utc)}

In [31]:
check_merges_df = pd.DataFrame([m.__dict__ for m in check_merges])

In [32]:
check_merges_df

,id,graph_name,merge_type,entity1_name,entity1_type,entity1_aliases,entity1_source_graph,entity2_name,entity2_type,entity2_aliases,canonical_name,similarity,reasoning,occurrence_count_after,created_at
0,f5da0770-e98a-419a-8cfd-eccb0584c7ca,book3_ch4,root,Julius Caesar,person,[Caesar],,Julius Caesar,person,[],Julius Caesar,None,,1,2026-03-15 20:29:47.668085+00:00
1,ab3743ba-8ecd-4863-91bf-8e72d35be6b1,book3_ch4,rule,Caesar,person,[],,Julius Caesar,person,[Caesar],Julius Caesar,None,,2,2026-03-15 20:29:47.668127+00:00
2,e2e25e48-ae02-4996-b551-73dfac6b47a1,book3_ch4,rule,Caesar,person,[],,Julius Caesar,person,[Caesar],Julius Caesar,None,,3,2026-03-15 20:29:47.668250+00:00
3,0606e63f-9347-4008-b95f-c156108ca699,book3_ch4,rule,Julius Caesar,person,[Caesar],,Julius Caesar,person,[Caesar],Julius Caesar,None,,4,2026-03-15 20:29:47.668313+00:00
4,549265b6-46a7-4e54-99ff-6bafd2d0164b,book3_ch4,rule,Caesar,person,[],,Julius Caesar,person,[Caesar],Julius Caesar,None,,5,2026-03-15 20:29:47.668428+00:00
5,b8ec5614-9607-4c67-b36f-987bfb8ae325,book3_ch4,rule,Julius Caesar,person,[Caesar],,Julius Caesar,person,[Caesar],Julius Caesar,None,,6,2026-03-15 20:29:47.668492+00:00
6,03e9a15b-985f-424f-bca1-c62a40643e8e,book3_ch4,rule,Julius Caesar,person,[],,Julius Caesar,person,[Caesar],Julius Caesar,None,,7,2026-03-15 20:29:47.668584+00:00
7,44ec805a-418e-4c8a-89c4-0d5c0e44535c,book3_ch4,rule,Julius Caesar,person,[Julius],,Julius Caesar,person,[Caesar],Julius Caesar,None,,8,2026-03-15 20:29:47.668695+00:00
8,7bd1759a-e37c-4c0b-aa63-908a23da2dd8,book3_ch4,rule,Julius Caesar,person,[Caesar],,Julius Caesar,person,"[Julius, Caesar]",Julius Caesar,None,,9,2026-03-15 20:29:47.670053+00:00
9,f7a69ff0-7e5f-4e06-9a23-afbb19686eeb,book3_ch4,rule,Julius Caesar,person,[],,Julius Caesar,person,"[Julius, Caesar]",Julius Caesar,None,,10,2026-03-15 20:29:47.670645+00:00


In [33]:
check_merges_2 = mgr.kg_merge_decisions.find_by_entity("Augustus", entity_type="person", graph_name="book3_ch4")

In [34]:
len(check_merges_2)

15

In [35]:
check_merges_2_df = pd.DataFrame([m.__dict__ for m in check_merges_2])

In [38]:
check_merges_2_df.sort_values("occurrence_count_after")

,id,graph_name,merge_type,entity1_name,entity1_type,entity1_aliases,entity1_source_graph,entity2_name,entity2_type,entity2_aliases,canonical_name,similarity,reasoning,occurrence_count_after,created_at
9,a064e59e-1a21-4dbe-8dd5-570f51e1d6f7,book3_ch4,root,Augustus,person,[],,Augustus,person,[],Augustus,None,,1,2026-03-15 20:29:47.671595+00:00
0,1ff68b74-0fe9-4a41-ad35-6c512141ce7c,book3_ch4,rule,Octavian,person,"[Caesar, Augustus]",,Octavian,person,[Caesar Augustus],Augustus,None,,2,2026-03-15 20:29:47.668672+00:00
10,8dec6d1b-fcc1-400b-be8d-6f9633e2a533,book3_ch4,rule,Augustus,person,[],,Augustus,person,[],Augustus,None,,2,2026-03-15 20:29:47.673466+00:00
4,6b0e48b3-6918-4f80-aea3-786b3c9790f1,book3_ch4,rule,Octavian,person,"[Augustus, princeps]",,Augustus,person,"[Octavian, Caesar Augustus, Caesar]",Augustus,None,,3,2026-03-15 20:29:47.669867+00:00
11,1a9bc028-cc44-4fda-9aeb-cee27ee40934,book3_ch4,rule,Augustus,person,[],,Augustus,person,[],Augustus,None,,3,2026-03-15 20:29:47.673610+00:00
1,919c9bf8-9ff1-47ea-9f32-f8db6ce5a5c3,book3_ch4,rule,Augustus,person,[],,Augustus,person,"[Octavian, princeps, Caesar Augustus, Caesar]",Augustus,None,,4,2026-03-15 20:29:47.669947+00:00
12,825897c8-1b23-431e-9ff7-ae55eef6e14f,book3_ch4,rule,Augustus,person,[],,Augustus,person,[],Augustus,None,,4,2026-03-15 20:29:47.674017+00:00
2,ca42cfb5-06db-49db-ab35-ca083af01f1e,book3_ch4,rule,Augustus,person,[Caesar],,Augustus,person,"[Octavian, princeps, Caesar Augustus, Caesar]",Augustus,None,,5,2026-03-15 20:29:47.670037+00:00
13,0ffe2d48-692e-4008-b9f8-5e9e47db1820,book3_ch4,rule,Augustus,person,[],,Augustus,person,[],Augustus,None,,5,2026-03-15 20:29:47.674611+00:00
3,45b563f1-2b56-4511-a0fb-222a973de096,book3_ch4,rule,Augustus,person,[],,Augustus,person,"[Octavian, princeps, Caesar Augustus, Caesar]",Augustus,None,,6,2026-03-15 20:29:47.670123+00:00


In [12]:
greece_merges_ch2 = mgr.kg_merge_decisions.find_by_entity_name("Greece",graph_name="book3_ch2")

In [13]:
greece_merges_ch2

[KGMergeDecision(id='82ebd5c4-5b59-4418-9c6f-681841b43562', graph_name='book3_ch2', merge_type='rule', entity1_name='Greece', entity1_type='polity', entity1_aliases=['archaic Greece', 'classical Greece'], entity2_name='Greek world', entity2_type='polity', entity2_aliases=['Greece'], canonical_name='Greece', similarity=None, reasoning='', created_at=datetime.datetime(2026, 3, 14, 23, 53, 0, 436961, tzinfo=datetime.timezone.utc)),
 KGMergeDecision(id='ebd376ed-a222-4705-a716-581c432bf03a', graph_name='book3_ch2', merge_type='llm', entity1_name='early Greece', entity1_type='polity', entity1_aliases=[], entity2_name='Greeks', entity2_type='polity', entity2_aliases=['Hellenes'], canonical_name='Greeks (early Greece)', similarity=0.7045108821057713, reasoning="Both mentions refer to the same broad historical entity: the Greeks/early Greek civilization in the archaic period (before and around 500 BC). 'early Greece' describes the society and colonizing phase of the Greeks, while 'Greeks' (Hel

In [23]:
greece_merges_ch2[1].__dict__

{'id': 'ebd376ed-a222-4705-a716-581c432bf03a',
 'graph_name': 'book3_ch2',
 'merge_type': 'llm',
 'entity1_name': 'early Greece',
 'entity1_type': 'polity',
 'entity1_aliases': [],
 'entity2_name': 'Greeks',
 'entity2_type': 'polity',
 'entity2_aliases': ['Hellenes'],
 'canonical_name': 'Greeks (early Greece)',
 'similarity': 0.7045108821057713,
 'reasoning': "Both mentions refer to the same broad historical entity: the Greeks/early Greek civilization in the archaic period (before and around 500 BC). 'early Greece' describes the society and colonizing phase of the Greeks, while 'Greeks' (Hellenes) is the people themselves; here both clearly point to the same cultural-political group in ancient Greece. They are not distinct entities (one is not a specific subgroup or different polity).",
 'created_at': datetime.datetime(2026, 3, 14, 23, 53, 0, 437003, tzinfo=datetime.timezone.utc)}

In [20]:
greece_merges_ch2_df = pd.DataFrame([m.__dict__ for m in greece_merges_ch2])

In [22]:
greece_merges_ch2_df['merge_type'].value_counts()

merge_type
llm     22
rule     9
Name: count, dtype: int64